# MTCG: Multi-Template Computational Graph for Traffic Demand Flow Estimation

**Melbourne Network — Model Training**

**Authors:**
- Xin (Bruce) Wu, Department of Civil and Environmental Engineering, Villanova University, USA
- Feng Shao, School of Mathematics, China University of Mining and Technology, China

**Contact:** xwu03@villanova.edu

---

MIT License

Copyright (c) 2026 Xin (Bruce) Wu, Feng Shao

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# MTCG Model Training — Melbourne Network

This notebook trains the Multi-Template Computational Graph (MTCG) model on the Melbourne transportation network.

In [1]:
import networkx as nx
import copy as cp
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from pandas import DataFrame
import scipy.sparse as sp
import math
import time

plt.rc('font',family='Times New Roman')

import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
import torch.nn.functional as F
import torch.optim as optim

## 1. Data Loading

Load network topology, OD pairs, link attributes, and path data for all templates.

In [2]:
od_pair = pd.read_csv('data_new_1/OD_pair.csv').values
num_od = len(od_pair)
od_pair.shape

(25850, 2)

In [3]:
link_attributes = pd.read_csv('data_new_1/link.csv')
link_attributes = link_attributes[['link_id', 'from_node_id', 'to_node_id','capacity', 'VDF_fftt1']]
link_attributes.loc[link_attributes['VDF_fftt1']==100, 'VDF_fftt1']=0
num_link = len(link_attributes)
link_attributes

,link_id,from_node_id,to_node_id,capacity,VDF_fftt1
0,1,1,2,9000.0,0.102812
1,2,3,4,9000.0,0.089317
2,3,5,6,1800.0,0.068867
3,4,8,9,1000.0,0.046286
4,5,10,8,950.0,0.189784
...,...,...,...,...,...
7114,7115,2348,1936,13000.0,1.000000
7115,7116,2293,2348,13000.0,1.000000
7116,7117,2294,2348,13000.0,1.000000
7117,7118,2287,2348,13000.0,1.000000


In [4]:
# free flow travel time
free_flow_tt = link_attributes['VDF_fftt1'].values
free_flow_tt = torch.from_numpy(free_flow_tt).to(torch.float32)

# link capacity
link_capacity = link_attributes['capacity'].values
link_capacity = torch.from_numpy(link_capacity).to(torch.float32)

In [5]:
# VDF theta parameter, uniformly set to 2.5, device/dtype consistent with free_flow_tt
vdf_theta = torch.full_like(free_flow_tt, 2.5)

In [6]:
k = 3

path1 = pd.read_csv('data_new_1/agent_rep.csv')
path1 = path1[['o_zone_id', 'd_zone_id', 'path_id', 'link_sequence']]
path1 = path1[path1['path_id']<k]
path1.reset_index(drop=True, inplace=True)

path_count1 = path1.groupby(['o_zone_id', 'd_zone_id']).count().reset_index()

p1 = path_count1[path_count1['path_id']==1]
p1.reset_index(drop=True, inplace=True)
path_add_1 = DataFrame()
for i in range(len(p1)):
    path_add_1 = pd.concat([path_add_1,path1[(path1['o_zone_id']==p1.loc[i,'o_zone_id'])&
                                             (path1['d_zone_id']==p1.loc[i,'d_zone_id'])]])

p2 = path_count1[path_count1['path_id']==2]
p2.reset_index(drop=True, inplace=True)
path_add_2 = DataFrame()
for i in range(len(p2)):
    path_add_2 = pd.concat([path_add_2,path1[(path1['o_zone_id']==p2.loc[i,'o_zone_id'])&
                                             (path1['d_zone_id']==p2.loc[i,'d_zone_id'])&
                                             (path1['path_id']==0)]])

path_last1 = pd.concat([path1, path_add_1, path_add_1, path_add_2])
path_last1 = path_last1.sort_values(['o_zone_id', 'd_zone_id', 'path_id'])
path_last1.reset_index(drop=True, inplace=True)
num_path = len(path_last1)

LP1 = np.zeros([num_path, num_link])
for i in range(num_path):
    
    path_index = np.array(list(map(int, path_last1.loc[i, 'link_sequence'].split(';'))))-1
    LP1[i, path_index] = 1
        
LP1 = torch.from_numpy(LP1.T).to(torch.float32)
LP1.shape

torch.Size([7119, 77550])

In [7]:
path2 = pd.read_csv('data_new_2/agent_rep.csv')
path2 = path2[['o_zone_id', 'd_zone_id', 'path_id', 'link_sequence']]
path2 = path2[path2['path_id']<k]
path2.reset_index(drop=True, inplace=True)

path_count2 = path2.groupby(['o_zone_id', 'd_zone_id']).count().reset_index()

p1 = path_count2[path_count2['path_id']==1]
p1.reset_index(drop=True, inplace=True)
path_add_1 = DataFrame()
for i in range(len(p1)):
    path_add_1 = pd.concat([path_add_1,path2[(path2['o_zone_id']==p1.loc[i,'o_zone_id'])&
                                             (path2['d_zone_id']==p1.loc[i,'d_zone_id'])]])

p2 = path_count2[path_count2['path_id']==2]
p2.reset_index(drop=True, inplace=True)
path_add_2 = DataFrame()
for i in range(len(p2)):
    path_add_2 = pd.concat([path_add_2,path2[(path2['o_zone_id']==p2.loc[i,'o_zone_id'])&
                                             (path2['d_zone_id']==p2.loc[i,'d_zone_id'])&
                                             (path2['path_id']==0)]])

path_last2 = pd.concat([path2, path_add_1, path_add_1, path_add_2])
path_last2 = path_last2.sort_values(['o_zone_id', 'd_zone_id', 'path_id'])
path_last2.reset_index(drop=True, inplace=True)

LP2 = np.zeros([num_path, num_link])
for i in range(num_path):
    
    path_index = np.array(list(map(int, path_last2.loc[i, 'link_sequence'].split(';'))))-1
    LP2[i, path_index] = 1
        
LP2 = torch.from_numpy(LP2.T).to(torch.float32)
LP2.shape

torch.Size([7119, 77550])

In [8]:
path3 = pd.read_csv('data_new_3/agent_rep.csv')
path3 = path3[['o_zone_id', 'd_zone_id', 'path_id', 'link_sequence']]
path3 = path3[path3['path_id']<k]
path3.reset_index(drop=True, inplace=True)

path_count3 = path3.groupby(['o_zone_id', 'd_zone_id']).count().reset_index()

p1 = path_count3[path_count3['path_id']==1]
p1.reset_index(drop=True, inplace=True)
path_add_1 = DataFrame()
for i in range(len(p1)):
    path_add_1 = pd.concat([path_add_1,path3[(path3['o_zone_id']==p1.loc[i,'o_zone_id'])&
                                             (path3['d_zone_id']==p1.loc[i,'d_zone_id'])]])

p2 = path_count3[path_count3['path_id']==2]
p2.reset_index(drop=True, inplace=True)
path_add_2 = DataFrame()
for i in range(len(p2)):
    path_add_2 = pd.concat([path_add_2,path3[(path3['o_zone_id']==p2.loc[i,'o_zone_id'])&
                                             (path3['d_zone_id']==p2.loc[i,'d_zone_id'])&
                                             (path3['path_id']==0)]])

path_last3 = pd.concat([path3, path_add_1, path_add_1, path_add_2])
path_last3 = path_last3.sort_values(['o_zone_id', 'd_zone_id', 'path_id'])
path_last3.reset_index(drop=True, inplace=True)

LP3 = np.zeros([num_path, num_link])
for i in range(num_path):
    
    path_index = np.array(list(map(int, path_last3.loc[i, 'link_sequence'].split(';'))))-1
    LP3[i, path_index] = 1
        
LP3 = torch.from_numpy(LP3.T).to(torch.float32)
LP3.shape

torch.Size([7119, 77550])

In [9]:
LP0 = torch.stack([LP1,LP2,LP3])
LP0.shape

torch.Size([3, 7119, 77550])

In [10]:
# demand_6_10 = pd.read_csv('data_new_1/demand_6-10.csv').values.T
# demand_6_10 = torch.from_numpy(demand_6_10).to(torch.float32)
# demand_6_10.shape

## 2. Demand Generation

Load and mix demand from three scenario templates using predefined weights.

In [11]:
# ====== Load demand from three scenario templates (transposed shape: (16, num_od)) ======
d1 = pd.read_csv('data_new_1/demand_6-10_seeded.csv').values.T
d2 = pd.read_csv('data_new_2/demand_6-10_seeded.csv').values.T
d3 = pd.read_csv('data_new_3/demand_6-10_seeded.csv').values.T

assert d1.shape == d2.shape == d3.shape, "Demand dimensions inconsistent across three templates"
T_total, num_od_check = d1.shape

W = np.zeros((T_total, 3), dtype=float)

W = np.zeros((16,3))
W[0] = [0.9, 0.05, 0.05]  
W[1] = [0.9, 0.05, 0.05]  
W[2] = [0.8, 0.1, 0.1]  
W[3] = [0.7, 0.2, 0.1]  

W[4] = [0.2, 0.4, 0.4] 
W[5] = [0.3, 0.5, 0.2]  
W[6] = [0.3, 0.6, 0.1]  
W[7] = [0.2, 0.6, 0.2]  

W[8] = [0.2, 0.6, 0.2]  
W[9] = [0.1, 0.7, 0.2]  
W[10] = [0.1, 0.8, 0.1]  
W[11] = [0.2, 0.6, 0.2]  

W[12] = [0.3, 0.3, 0.4]  
W[13] = [0.3, 0.4, 0.3]  
W[14] = [0.3, 0.4, 0.3] 
W[15] = [0.3, 0.4, 0.3]  

# Normalize weights (in case they do not sum to exactly 1)
row_sum = W.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
W = W / row_sum

# ====== Compute weighted mixture demand per timestep ======
demand_mix = W[:, [0]] * d1 + W[:, [1]] * d2 + W[:, [2]] * d3   # (16, num_od)

# Convert to torch tensor for downstream use
demand_6_10 = torch.from_numpy(demand_mix.astype(np.float32))
print("demand_6_10.shape =", demand_6_10.shape)  # Expected: (16, num_od)


demand_6_10.shape = torch.Size([16, 25850])


In [12]:
demand_6_10.sum(axis=1)

tensor([15708.9004, 16932.9004, 17606.7988, 19113.0000, 25808.1992, 29719.1016,
        23949.2988, 23388.0020, 29859.0000, 29761.0996, 28941.4980, 28699.0020,
        21957.0996, 20422.4961, 28122.8008, 18770.8008])

In [13]:
flow = pd.read_csv('data_new/link_flow.csv')
link_flow = flow.values
link_flow = torch.from_numpy(link_flow).to(torch.float32)
link_flow.shape

torch.Size([16, 448])

## 3. Data Preparation

Reshape data into tensors and split into training/test sets.

In [14]:
# Timestep
T = 12
num_sample = 16 - T + 1
num_sample

5

In [15]:
input_set = torch.zeros(num_sample, num_od)
for i in range(num_sample):
    input_set[i] = torch.sum(demand_6_10[i:i+T],axis=0)
input_set.shape

torch.Size([5, 25850])

In [16]:
output_set = torch.zeros(num_sample, T, link_flow.shape[1])
for i in range(num_sample):
    output_set[i] = link_flow[i:i+T]
output_set = output_set.transpose(0,1)
output_set.shape

torch.Size([12, 5, 448])

In [17]:
demand_gt = torch.zeros(num_sample, T, num_od)
for i in range(num_sample):
    demand_gt[i] = demand_6_10[i:i+T]
demand_gt = demand_gt.transpose(0,1)
demand_gt.shape

torch.Size([12, 5, 25850])

In [18]:
# train set
train_input = input_set[:4,:]
train_output = output_set[:,:4,:]

# test set
test_input = input_set[4:,:]
test_output = output_set[:,4:,:]

print(train_input.shape)
print(train_output.shape)
print(test_input.shape)
print(test_output.shape)

torch.Size([4, 25850])
torch.Size([12, 4, 448])
torch.Size([1, 25850])
torch.Size([12, 1, 448])


In [19]:
demand_reference = demand_gt[:,4:,:]
demand_reference.shape

torch.Size([12, 1, 25850])

## 4. Model Components

Define the BPR volume-delay function, neural network layers, and attention mechanism.

In [20]:
# # Define the BPR function
# def BPR(link_flow, free_flow_tt, link_capacity):
#     link_time = free_flow_tt * (1 + 0.15 * (link_flow/link_capacity)**4)
#     return link_time

In [21]:
def BPR(link_flow, free_flow_tt, link_capacity):
    denom = torch.clamp(link_capacity * vdf_theta, min=1e-6)  # (num_link,)
    x = link_flow / denom                                    # Shape auto-broadcasts
    return free_flow_tt * (1.0 + 0.15 * x**4)

In [22]:
# Given theta, path travel times, and the link-path incidence matrix, 
# the OD demands are distributed to obtain link flows and link travel times.
def assignment(demand, theta, path_time, LP):
    
    batch_size = demand.shape[0]
    
    # The OD demand assignment for the first timestep is based on the free flow travel time, while for subsequent timesteps, 
    # it is allocated based on the travel time from the previous timestep.
    if path_time.dim()==1:
        path_time = path_time.repeat(batch_size,1)
    
    # Reshaping the demand to have the same dimension as the path choice proportion matrix so that they can be multiplied.
    demand_expand = demand.reshape(-1,1).repeat(1,k).reshape(batch_size,-1) # shape: batch_size × num_path
    
     # Reshape the theta
    theta_expand = theta.repeat(k,1).transpose(0,1).reshape(1,-1)   # shape: 1 × num_path
    
    # Calculate e^(-theta*path_time)
    tp0 = (-path_time * theta_expand).reshape(batch_size,-1,k)  #shape: batch_size×num_OD×k
    tp = math.e ** tp0  #shape: batch_size×num_OD×k
    
    # Set a lower bound for e^(-theta*path_time)
    P0 = torch.max(tp, (10**-10)*torch.ones(tp.shape)) #shape: batch_size×num_OD×k

    # Sum over the last dimension
    P0_sum = torch.sum(P0,dim=2).view(batch_size,-1,1)  #shape: batch_size×num_OD×1

    # Calculate the path choice proportion
    P0 = P0/P0_sum #shape: batch_size×num_OD×k

    P = P0.view(batch_size,-1)  # shape: batch_size × num_path

    path_flow = demand_expand * P  # shape: batch_size × num_path

    # Estimate link flow and link time
    link_flow = torch.mm(path_flow,LP.T)   # shape: batch_size × num_link
    link_time = BPR(link_flow, free_flow_tt, link_capacity)
    
    return link_flow,link_time

In [23]:
def Initialization(parameters):
    for para in parameters:
        nn.init.kaiming_normal_(para)

In [24]:
# Calculate the attention coefficients between estimated link flows from each channel and input OD demands, 
# to obtain weighted values for link flows.

# Hyperparameter: hidden_size, the dimensions of the embedding tensor
class Attention(nn.Module):
    
    def __init__(self, num_od, num_link, hidden_size):
        
        super(Attention, self).__init__()
    
        #self.linear_demand = nn.Linear(num_od, hidden_size,bias = False)
        self.weights_demand = Parameter(torch.FloatTensor(num_od, hidden_size))
        self.weights_linkflow = Parameter(torch.FloatTensor(T, num_link, hidden_size))
        self.hidden_size = hidden_size
        
        Initialization([self.weights_demand, self.weights_linkflow])
    
    def forward(self, demand, X):
        
        batch_size = len(demand) # demand shape: (batch_size, num_od)
        
        q = torch.nn.LayerNorm(self.hidden_size)(torch.mm(demand, self.weights_demand)) # (batch_size, hidden_size)
        q = q.reshape(1,batch_size,1,-1) # shape: (1, batch size, 1, hidden_size)
        q = q.repeat(T,1,1,1) # shape: (T, batch size, 1, hidden_size)
        
        w = self.weights_linkflow.reshape(T, 1, num_link, -1) # shape: (T, 1, num_link, hidden_size)
        w = w.repeat(1, batch_size, 1, 1) # shape: (T, batch size, num_link, hidden_size)
        
        # X.shape: (S, T, batch size, num_link)
        X = X.permute(1,2,0,3) # shape: (T, batch size, S, num_link)
        
        x =  torch.nn.LayerNorm(self.hidden_size)(torch.matmul(X, w))  # shape: (T, batch size, S, hidden_size)  
        
        scores = torch.matmul(q, x.transpose(2, 3))/math.sqrt(x.shape[-1]) # (T, batch size, 1, S)
        
        att = F.softmax(scores, dim=-1) # (T, batch size, 1, S)
        
        x_output = torch.matmul(att,X).reshape(T, batch_size,-1) # (T, batch size, 1, num_link)-->(T, batch size, num_link)
        
        return att, x_output

## 5. Model Definition

Define the MTCG model class with multi-template architecture and attention-based fusion.

In [25]:
class MRLN(Module):
    
    # hidden_layer: the number of neurons in the hidden layer of the DNN that 
    #               maps the link flows from the previous timestep to the OD demands in the next timestep.
    # att_hidden_size:the dimensions of the embedding tensor for attention calculation
    # hidden_layer2: the number of neurons in the hidden layer of the DNN that maps the initial OD demand 
    #                to the proportion of OD demands for the first timestep.

    def __init__(self, num_link, num_od, hidden_layer, att_hidden_size, hidden_layer2):
        
        super(MRLN, self).__init__()
        
        self.num_od = num_od
        
        # Same channel, same timestep, and same OD pair share a common theta
        self.theta = Parameter(torch.FloatTensor(S, T, num_od))
        nn.init.xavier_uniform_(self.theta)
        
        # For each channel, ODME is performed using a three-layer DNN
        # Different timesteps share weights and biases
        
        # flow network
        self.weights1_v = Parameter(torch.FloatTensor(S, num_link, hidden_layer[0]))
        self.weights2_v = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
        self.weights3_v = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        self.bias1_v = Parameter(torch.FloatTensor(S, hidden_layer[0]))
        self.bias2_v = Parameter(torch.FloatTensor(S, hidden_layer[1]))
        self.bias3_v = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights1_v, self.weights2_v, self.weights3_v, self.bias1_v, self.bias2_v, self.bias3_v])
        
        # od travel time network
        self.weights1_t = Parameter(torch.FloatTensor(S, num_od, hidden_layer[0]))
        self.weights2_t = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
        self.weights3_t = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        self.bias1_t = Parameter(torch.FloatTensor(S, hidden_layer[0]))
        self.bias2_t = Parameter(torch.FloatTensor(S, hidden_layer[1]))
        self.bias3_t = Parameter(torch.FloatTensor(S, num_od))
        
        Initialization([self.weights1_t, self.weights2_t, self.weights3_t, self.bias1_t, self.bias2_t, self.bias3_t])
              
        
        
        #################
        
        # flow network 2
        # self.weights1_v2 = Parameter(torch.FloatTensor(S, num_od, num_od))
        # self.weights2_v2 = Parameter(torch.FloatTensor(S, hidden_layer[0], hidden_layer[1]))
        # self.weights3_v2 = Parameter(torch.FloatTensor(S, hidden_layer[1], num_od))
        
        # self.bias1_v2 = Parameter(torch.FloatTensor(S, num_od))
        # self.bias2_v2 = Parameter(torch.FloatTensor(S, hidden_layer[1]))
        # self.bias3_v2 = Parameter(torch.FloatTensor(S, num_od))
        
        # Initialization([self.weights1_v2, self.weights2_v2, self.weights3_v2, self.bias1_v2, self.bias2_v2, self.bias3_v2])
        
        
        ##################
        
        # Attention layer
        self.attention = Attention(num_od, num_link, att_hidden_size)   
        
        # The initial allocation of OD demands using a three-layer DNN
        self.weights_d1 = Parameter(torch.FloatTensor(S, num_od, hidden_layer2[0]))
        self.weights_d2 = Parameter(torch.FloatTensor(S, hidden_layer2[0], hidden_layer2[1]))
        self.weights_d3 = Parameter(torch.FloatTensor(S, hidden_layer2[1], T * num_od))
        
        self.bias_d1 = Parameter(torch.FloatTensor(S, hidden_layer2[0]))
        self.bias_d2 = Parameter(torch.FloatTensor(S, hidden_layer2[1]))
        self.bias_d3 = Parameter(torch.FloatTensor(S, T * num_od))
        
        Initialization([self.weights_d1, self.weights_d2, self.weights_d3, self.bias_d1, self.bias_d2, self.bias_d3])


    def forward(self, demand):
        
        batch_size = demand.shape[0]
        
        # VOT
        theta = torch.abs(self.theta)
        
        # For storing link flows and OD demands
        link_flow = torch.zeros(S, T, batch_size, num_link)
        demand_pred = torch.zeros(S, T, batch_size, num_od) 
        demand_sum = torch.zeros(S, batch_size, num_od)
        
        for s in range(S):
            
            demand_proportion = torch.nn.LayerNorm(num_od)(demand)
            
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d1[s]) + self.bias_d1[s], negative_slope=0.2)
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d2[s]) + self.bias_d2[s], negative_slope=0.2)
            demand_proportion = F.leaky_relu(torch.mm(demand_proportion, self.weights_d3[s]) + self.bias_d3[s], negative_slope=0.2)

            demand_proportion = demand_proportion.reshape(batch_size, self.num_od, T)
#             demand_proportion = torch.nn.LayerNorm(T)(demand_proportion)
            demand_proportion = torch.softmax(demand_proportion, axis=2)  # (batch size, num_od, T)

            demand_initial = demand_proportion * demand.reshape(batch_size, self.num_od, 1)  # (batch size, num_od, T)
            demand_initial = demand_initial.permute(2,0,1)   # (T, batch size, num_od)
        
            # th 1th timestep
            demand_st = demand_initial[0]   # shape:(batch size, num_od)
            theta_st = theta[0, 0]
            
            LP_s = LP[s]
            
            link_time_st = free_flow_tt.repeat(batch_size,1)
            path_time_st = torch.mm(link_time_st,LP_s)
            
            link_flow_st,link_time_st = assignment(demand_st, theta_st, path_time_st, LP_s)
            
            # Calculate the od travel time
            path_time_st = torch.mm(link_time_st,LP_s) # shape:(batch size, num_path)
            od_travel_time = torch.mean(path_time_st.reshape(batch_size, num_od, k),dim=-1) # shape:(batch size, num_od)
            
            link_flow_s = torch.zeros(T, batch_size, num_link)
            link_flow_s[0] = link_flow_st
            
            demand_s = torch.zeros(T, batch_size, num_od)
            demand_s[0] = demand_st
            
            demand_sum_s = demand_st
            
            for t in range(T-1):
                                
                #########
                demand_st_v = torch.nn.LayerNorm(num_link)(link_flow_st)                
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights1_v[s]) + self.bias1_v[s])
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights2_v[s]) + self.bias2_v[s])
                demand_st_v = torch.tanh(torch.mm(demand_st_v, self.weights3_v[s]) + self.bias3_v[s])
                
                demand_st_t = torch.nn.LayerNorm(num_od)(od_travel_time)                
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights1_t[s]) + self.bias1_t[s])
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights2_t[s]) + self.bias2_t[s])
                demand_st_t = torch.tanh(torch.mm(demand_st_t, self.weights3_t[s]) + self.bias3_t[s])
                
                demand_st0 = 0.5 * (demand_st_v + demand_st_t)
                
                ##############
                
#                 demand_st_v2 = torch.mm(demand_st, self.weights1_v2[s]) + self.bias1_v2[s]
#                 demand_st_v2 = F.leaky_relu(demand_st_v2, negative_slope=0.2)
                
                
                demand_st = (1 + 0.5*demand_st0) * demand_st 
#             + demand_st_v2
                
                # The second source of demand_st, obtained by evenly distributing the total demand
                
                # Assignment
                theta_st = theta[0,0]
                link_flow_st,link_time_st = assignment(demand_st, theta_st, path_time_st, LP_s)
                
                # Update the path time and od travel time
                path_time_st = torch.mm(link_time_st,LP_s)
                od_travel_time = torch.mean(path_time_st.reshape(batch_size, num_od, k),dim=-1)
                
                link_flow_s[t+1] = link_flow_st
                demand_s[t+1] = demand_st
                
                demand_sum_s = demand_sum_s + demand_st
            
            link_flow[s] = link_flow_s
            demand_pred[s] = demand_s
            
            demand_sum[s] = demand_sum_s
        
        att, link_flow = self.attention(demand,link_flow)
        
        # att shape: (T, batch size, 1, S)
        # demand_pred shape: (S, T, batch_size, num_od)
        demand_pred = torch.matmul(att,demand_pred.permute(1,2,0,3)).reshape(T, batch_size,-1)
        
        att = att.reshape(T, batch_size, S)
        
        return att, demand_sum, demand_pred, link_flow

In [26]:
S = 3
LP= LP0[:S]
# path_time_initial = torch.sum(LP.transpose(2,1) * free_flow_tt, axis=2)
# path_time_initial.shape

In [27]:
# Initialize MRLN
mrln = MRLN(num_link=num_link, num_od=num_od,hidden_layer=[128,128],att_hidden_size=128, hidden_layer2=[128,128])
# View training parameters
for name, parameter in mrln.named_parameters():
    print(name, parameter.shape)

theta torch.Size([3, 12, 25850])
weights1_v torch.Size([3, 7119, 128])
weights2_v torch.Size([3, 128, 128])
weights3_v torch.Size([3, 128, 25850])
bias1_v torch.Size([3, 128])
bias2_v torch.Size([3, 128])
bias3_v torch.Size([3, 25850])
weights1_t torch.Size([3, 25850, 128])
weights2_t torch.Size([3, 128, 128])
weights3_t torch.Size([3, 128, 25850])
bias1_t torch.Size([3, 128])
bias2_t torch.Size([3, 128])
bias3_t torch.Size([3, 25850])
weights_d1 torch.Size([3, 25850, 128])
weights_d2 torch.Size([3, 128, 128])
weights_d3 torch.Size([3, 128, 310200])
bias_d1 torch.Size([3, 128])
bias_d2 torch.Size([3, 128])
bias_d3 torch.Size([3, 310200])
attention.weights_demand torch.Size([25850, 128])
attention.weights_linkflow torch.Size([12, 7119, 128])


## 6. Training Setup

Configure loss functions, optimizer (Adam), and hyperparameters.

In [28]:
mse_loss = nn.MSELoss()
l1_loss = nn.L1Loss()
optimizer = torch.optim.Adam(mrln.parameters(), lr=0.0002, betas=(0.5, 0.999))
# batch_size = 4
n_epochs = 60
sample_size = 7

In [29]:
# observable link numbers
# observation_link_number = flow.columns.values.astype(np.float).astype(int)-1
observation_link_number = flow.columns.values.astype(float).astype(int) - 1

num_observed_link = len(observation_link_number)

# randomly select unobservable link numbers
unobserved_link_idx = np.random.choice(num_observed_link, 45, replace=False)
unobserved_link = observation_link_number[unobserved_link_idx]

observed_link_idx = np.delete(np.arange(num_observed_link),unobserved_link_idx)
observed_link = observation_link_number[observed_link_idx]

In [30]:
loss_test_linkflow = torch.zeros(n_epochs)
loss_test_demand = torch.zeros(n_epochs)
loss_test_vdf = torch.zeros(n_epochs)

loss_train = torch.zeros(n_epochs)

def train(n_epochs):
    w1, w2, w3 = 1, 0.5, 0.001
    t0 = time.time()
    for epoch in range(n_epochs):
    
        
        mrln.zero_grad()

        # Input
        demand_input = train_input
        link_flow_output = train_output        
        
        # Output
        att_pred, demand_sum, demand_pred, link_flow_pred = mrln(demand_input)
        
        # link flow loss
        flow_with_sensors = link_flow_output[:,:,observed_link_idx]
        flow_pred_with_sensors = link_flow_pred[:,:,observed_link]
        
        link_flow_loss = mse_loss(flow_pred_with_sensors,flow_with_sensors)
        
        # demand loss
        demand_loss = mse_loss(demand_sum, demand_input.reshape(1,demand_input.shape[0],-1).repeat(S,1,1))

        
        # vdf_loss
        vdf_estimate = link_flow_pred * free_flow_tt + 0.03*(link_flow_pred**5)*free_flow_tt/(link_capacity**4)
        vdf_loss = torch.mean(torch.sum(vdf_estimate, dim=-1))
        
        total_loss = w1 * link_flow_loss + w2 * demand_loss + w3 * vdf_loss
        
        ###########
        att_test_pred, demand_test_sum_pred, demand_test_pred, link_flow_test_pred = mrln(test_input)
        
        loss_test_linkflow[epoch] = mse_loss(test_output, link_flow_test_pred[:,:,observation_link_number])
        loss_test_demand[epoch] = mse_loss(demand_reference, demand_test_pred)
        
        vdf_test_estimate = link_flow_test_pred * free_flow_tt + 0.03*(link_flow_test_pred**5)*free_flow_tt/(link_capacity**4)
        loss_test_vdf[epoch] = torch.mean(torch.sum(vdf_test_estimate, dim=-1))
        
        loss_train[epoch] = total_loss
        
        ###########

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters = mrln.parameters(), max_norm=1, norm_type=2)
        optimizer.step()
        
        print(
            "[Epoch %d/%d][link flow loss: %f][demand loss: %f][vdf loss: %f][total loss: %f][time: %f]"
            % (epoch, n_epochs, link_flow_loss.item(), demand_loss.item(), vdf_loss.item(), total_loss.item(), time.time()-t0)
        )

## 7. Model Training

Execute the training loop with gradient clipping and epoch-wise loss tracking.

In [31]:
train(n_epochs)

[Epoch 0/60][link flow loss: 23857.873047][demand loss: 1.326241][vdf loss: 152599.453125][total loss: 24011.136719][time: 18.662847]
[Epoch 1/60][link flow loss: 24796.412109][demand loss: 1.955512][vdf loss: 154635.593750][total loss: 24952.025391][time: 37.626679]
[Epoch 2/60][link flow loss: 23747.931641][demand loss: 4.574757][vdf loss: 155342.515625][total loss: 23905.560547][time: 56.158241]
[Epoch 3/60][link flow loss: 22299.824219][demand loss: 12.421247][vdf loss: 155175.828125][total loss: 22461.210938][time: 75.257119]
[Epoch 4/60][link flow loss: 20904.412109][demand loss: 31.284243][vdf loss: 157192.296875][total loss: 21077.246094][time: 94.528247]
[Epoch 5/60][link flow loss: 19166.957031][demand loss: 72.014801][vdf loss: 161861.546875][total loss: 19364.826172][time: 113.488214]
[Epoch 6/60][link flow loss: 17182.335938][demand loss: 156.097931][vdf loss: 165562.750000][total loss: 17425.947266][time: 132.392963]
[Epoch 7/60][link flow loss: 15220.800781][demand loss:

## 8. Inference & Results

Generate predictions on the test set and save estimation results.

In [32]:
att_pred, demand_test_sum_pred, demand_test_pred, link_flow_test_pred = mrln(test_input)

In [33]:
DataFrame(att_pred.reshape(12,3).detach().numpy()).to_csv('att.csv',index=False)

In [34]:
DataFrame(mrln.theta[0,0].abs().detach().numpy()).to_csv('theta2.csv',index=False)

In [35]:
demand_test_pred.shape

torch.Size([12, 1, 25850])

In [36]:
demand_reference[0,0,:].max()

tensor(320.)

In [37]:
att_pred

tensor([[[0.5620, 0.3471, 0.0909]],

        [[0.3819, 0.4504, 0.1677]],

        [[0.1744, 0.5933, 0.2323]],

        [[0.1208, 0.5414, 0.3378]],

        [[0.1072, 0.4294, 0.4634]],

        [[0.1790, 0.2960, 0.5250]],

        [[0.2673, 0.1919, 0.5408]],

        [[0.4008, 0.1710, 0.4283]],

        [[0.5075, 0.1507, 0.3418]],

        [[0.6938, 0.0994, 0.2068]],

        [[0.7927, 0.1086, 0.0988]],

        [[0.8937, 0.0678, 0.0386]]], grad_fn=<ViewBackward0>)

In [38]:
mrln.theta[0,0].abs().mean()

tensor(0.0024, grad_fn=<MeanBackward0>)

In [39]:
DataFrame(mrln.theta[0,0].abs().detach().numpy()).to_csv('theta2.csv',index=False)

In [40]:
a = torch.sum(demand_test_pred,axis=-1).detach().numpy()

In [41]:
b = torch.sum(demand_reference,axis=-1).detach().numpy()

In [42]:
c = link_flow_test_pred[:,:,observation_link_number].sum(axis=-1).detach().numpy()

In [43]:
d = test_output.sum(axis=-1).detach().numpy()

In [44]:
e = np.hstack([a,b,c,d])
e

array([[ 30976.082,  25808.2  ,  91054.664,  96076.   ],
       [ 32381.336,  29719.102,  97227.32 , 108958.   ],
       [ 34331.676,  23949.299, 105738.945, 116355.   ],
       [ 35161.645,  23388.002, 107982.55 , 117743.   ],
       [ 35875.62 ,  29859.   , 108946.17 , 116901.   ],
       [ 36180.582,  29761.1  , 107668.46 , 115800.   ],
       [ 36551.78 ,  28941.498, 106579.41 , 112795.   ],
       [ 36596.21 ,  28699.002, 104900.05 , 110604.   ],
       [ 36862.594,  21957.1  , 103961.93 , 104375.   ],
       [ 36106.52 ,  20422.496,  98613.414, 104415.   ],
       [ 36203.22 ,  28122.8  ,  97260.234,  98318.   ],
       [ 36302.14 ,  18770.8  ,  94910.29 ,  93523.   ]], dtype=float32)

In [45]:
DataFrame(e).to_csv('estimation2.csv',index=False)

## 9. Loss Export

Save training and test loss trajectories to Excel files.

In [46]:
DataFrame(loss_train.detach().numpy()).to_excel('D:/Melbourne/Melbourne/loss_train.xlsx',index=False)

In [47]:
DataFrame((loss_test_linkflow+0.5*loss_test_demand+0.01*loss_test_vdf).detach().numpy()).to_excel('D:/Melbourne/Melbourne/loss_test.xlsx',index=False)

In [48]:
loss_test_demand

tensor([ 4.4978,  4.5314,  4.5852,  4.6831,  4.8785,  5.2427,  5.9138,  6.9951,
         8.2313,  9.2937, 10.0926, 11.0102, 11.8319, 12.0271, 11.4698, 11.6626,
        11.3276, 10.5210, 14.1950, 12.5267,  9.8581, 11.1628, 12.6010, 10.7654,
        11.0359, 11.9544, 10.3563, 11.9158, 10.4258, 11.7047, 10.6535, 11.7086,
        10.8419, 10.6235,  9.9556, 11.7373, 10.9364,  9.6651, 10.1517, 10.6326,
        10.9418,  9.6960, 10.5778, 10.0717, 11.0147,  9.9489, 11.0945, 10.1850,
        11.2248, 10.1928, 11.5280, 10.4560,  9.6559, 10.3752,  9.8940, 10.7867,
         9.9205, 10.7298, 10.0105, 11.0328], grad_fn=<CopySlices>)

In [49]:
test_output.shape

torch.Size([12, 1, 448])

In [50]:
link_flow_test_pred.shape

torch.Size([12, 1, 7119])

In [51]:
demand_test_pred.shape

torch.Size([12, 1, 25850])

In [52]:
Error = torch.zeros(4,3)
Error[0,0] = mse_loss(test_output[:,:,observed_link_idx], link_flow_test_pred[:,:,observed_link])**0.5
Error[0,1] = l1_loss(test_output[:,:,observed_link_idx], link_flow_test_pred[:,:,observed_link])
E1 = torch.abs(test_output[:,:,observed_link_idx]-link_flow_test_pred[:,:,observed_link])/test_output[:,:,observed_link_idx]
Error[0,2] = torch.mean(E1[E1<100])

Error[1,0] = mse_loss(test_output[:,:,unobserved_link_idx], link_flow_test_pred[:,:,unobserved_link])**0.5
Error[1,1] = l1_loss(test_output[:,:,unobserved_link_idx], link_flow_test_pred[:,:,unobserved_link])
E2 = torch.abs(test_output[:,:,unobserved_link_idx]-link_flow_test_pred[:,:,unobserved_link])/test_output[:,:,unobserved_link_idx]
Error[1,2] = torch.mean(E2[E2<100])

Error[2,0] = mse_loss(test_output, link_flow_test_pred[:,:,observation_link_number])**0.5
Error[2,1] = l1_loss(test_output, link_flow_test_pred[:,:,observation_link_number])
E3 = torch.abs(test_output-link_flow_test_pred[:,:,observation_link_number])/test_output
Error[2,2] = torch.mean(E3[E3<100])

Error[3,0] = mse_loss(demand_reference, demand_test_pred)**0.5
Error[3,1] = l1_loss(demand_reference, demand_test_pred)
E4 = torch.abs(demand_reference-demand_test_pred)/demand_reference
Error[3,2] = torch.mean(E4[E4<100])

Error = Error.detach().numpy()
Error[:,:2] = np.floor(100*Error[:,:2])/100
Error[:,2] = np.floor(10000*Error[:,2])/10000
Error = DataFrame(Error,index = ['With sensors', 'Without sensors', 'Total error','OD demand'], columns = ['RMSE', 'MAE', 'MAPE'])
Error

,RMSE,MAE,MAPE
With sensors,47.400002,31.820000,0.2079
Without sensors,119.739998,88.860001,0.5030
Total error,58.830002,37.549999,0.2376
OD demand,3.190000,1.350000,1.9579


## 10. Evaluation Metrics

Compute RMSE, MAE, and MAPE for link flow (with/without sensors) and OD demand.

In [53]:
def metrics(y_true, y_pred, mape_threshold=None, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]

    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    mae  = np.mean(np.abs(y_pred - y_true))

    # MAPE: computed only for |y_true| >= mape_threshold to avoid inflation by near-zero values
    if mape_threshold is None:
        den = np.clip(np.abs(y_true), eps, None)
        mape = np.mean(np.abs((y_pred - y_true)/den))
    else:
        mask = np.abs(y_true) >= mape_threshold
        if mask.any():
            den = np.clip(np.abs(y_true[mask]), eps, None)
            mape = np.mean(np.abs((y_pred[mask] - y_true[mask]) / den))
        else:
            mape = np.nan
    return rmse, mae, mape

# ---- Alignment: test_output columns are observed-column position indices; predictions use full-network linkID (0-based) ----
# Get full-network linkID (0-based) corresponding to observed columns from DataFrame
obs_ids_from_df = observation_link_number  # Array of observed link IDs

# Full set of position indices (0..num_obs-1)
all_pos = np.arange(test_output.shape[-1], dtype=int)
# Position indices for each subset
obs_pos   = observed_link_idx
unobs_pos = unobserved_link_idx
# Map subset position indices to full-network linkID (0-based) for prediction tensor indexing
all_ids   = obs_ids_from_df[all_pos]
obs_ids   = obs_ids_from_df[obs_pos]
unobs_ids = obs_ids_from_df[unobs_pos]

# ---- Extract data and compute metrics (squeeze batch dim when batch=1) ----
thr = 5.0  # Ground-truth threshold for MAPE; set to None to disable filtering

# With sensors
yt_obs = test_output[:, :, obs_pos].squeeze(1).detach().cpu().numpy()
yp_obs = link_flow_test_pred[:, :, obs_ids].squeeze(1).detach().cpu().numpy()
rmse_w, mae_w, mape_w = metrics(yt_obs, yp_obs, mape_threshold=thr)

# Without sensors
yt_unobs = test_output[:, :, unobs_pos].squeeze(1).detach().cpu().numpy()
yp_unobs = link_flow_test_pred[:, :, unobs_ids].squeeze(1).detach().cpu().numpy()
rmse_wo, mae_wo, mape_wo = metrics(yt_unobs, yp_unobs, mape_threshold=thr)

# Total (all observed columns)
yt_all = test_output[:, :, all_pos].squeeze(1).detach().cpu().numpy()
yp_all = link_flow_test_pred[:, :, all_ids].squeeze(1).detach().cpu().numpy()
rmse_all, mae_all, mape_all = metrics(yt_all, yp_all, mape_threshold=thr)

# OD demand
yt_od = demand_reference.squeeze(1).detach().cpu().numpy()
yp_od = demand_test_pred.squeeze(1).detach().cpu().numpy()
rmse_od, mae_od, mape_od = metrics(yt_od, yp_od, mape_threshold=thr)

# ---- Build table and keep variable name Error (backward compatible) ----
Error = np.array([
    [rmse_w,  mae_w,  mape_w],
    [rmse_wo, mae_wo, mape_wo],
    [rmse_all,mae_all,mape_all],
    [rmse_od, mae_od, mape_od],
], dtype=float)

# Pretty-print
Error_show = Error.copy()
Error_show[:,:2] = np.floor(100*Error_show[:,:2])/100
Error_show[:, 2] = np.floor(10000*Error_show[:,2])/10000

Error = pd.DataFrame(
    Error_show,
    index=['With sensors','Without sensors','Total error','OD demand'],
    columns=['RMSE','MAE','MAPE']
)
print(Error)
# ====================== Evaluation metrics (end of replacement block) ======================


                   RMSE    MAE    MAPE
With sensors      47.40  31.82  0.2067
Without sensors  119.74  88.86  0.5030
Total error       58.83  37.55  0.2365
OD demand          3.19   1.35  0.6931


In [54]:

import gzip, csv, math

def dump_pairs_simple(y_true_t, y_pred_t, out_csv_gz, sample_prob=0.02, thr=5.0):
    """
    Write (y_true, y_pred) to compressed CSV (gz) in small chunks,
    - without expanding everything into memory at once
    - keeping only points where y_true >= thr (avoid dense cluster near zero)
    - randomly sampling with probability sample_prob (default 2%)
    """
    kept = 0
    total = 0

    with gzip.open(out_csv_gz, "wt", newline="") as gz:
        w = csv.writer(gz)
        w.writerow(["y_true", "y_pred"])

        T, B = y_true_t.shape[0], y_true_t.shape[1]
        for t in range(T):
            for b in range(B):
                yt = y_true_t[t, b].detach().cpu().numpy().ravel()
                yp = y_pred_t[t, b].detach().cpu().numpy().ravel()

                m = np.isfinite(yt) & np.isfinite(yp)
                if thr is not None:
                    m &= (yt >= float(thr))

                yt = yt[m]; yp = yp[m]
                if yt.size == 0:
                    continue

                # Probabilistic sampling, simple and memory-efficient
                if sample_prob < 1.0:
                    sel = np.random.rand(yt.size) < sample_prob
                    yt = yt[sel]; yp = yp[sel]

                for a, b_ in zip(yt, yp):
                    w.writerow([float(a), float(b_)])
                kept += yt.size
                total += int(m.sum())

    print(f"[Saved] {out_csv_gz} | kept {kept} samples out of {total} eligible points")

# ============== Write the four estimation pairs to disk ==============

# 1) All observed links
dump_pairs_simple(
    test_output,                                      # (T, batch, #obs)
    link_flow_test_pred[:, :, observation_link_number],
    out_csv_gz="links_all_pairs.csv.gz",
    sample_prob=1,   # Sample 100%; adjust as needed
    thr=5.0             # Only store points with y_true>=5; set thr=None to store all
)

# 2) "With sensor" subset
dump_pairs_simple(
    test_output[:, :, observed_link_idx],
    link_flow_test_pred[:, :, observed_link],
    out_csv_gz="links_with_sensors_pairs.csv.gz",
    sample_prob=0.05,
    thr=5.0
)

# 3) "Without sensor" subset
dump_pairs_simple(
    test_output[:, :, unobserved_link_idx],
    link_flow_test_pred[:, :, unobserved_link],
    out_csv_gz="links_without_sensors_pairs.csv.gz",
    sample_prob=0.05,
    thr=5.0
)

# 4) OD demand
dump_pairs_simple(
    demand_reference,
    demand_test_pred,
    out_csv_gz="od_demand_pairs.csv.gz",
    sample_prob=1,
    thr=5.0
)


[Saved] links_all_pairs.csv.gz | kept 5372 samples out of 5372 eligible points
[Saved] links_with_sensors_pairs.csv.gz | kept 236 samples out of 4832 eligible points
[Saved] links_without_sensors_pairs.csv.gz | kept 22 samples out of 540 eligible points
[Saved] od_demand_pairs.csv.gz | kept 18047 samples out of 18047 eligible points
